<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold3_of3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold 3 (OpenFold3)

Predict protein, RNA, DNA, and small-molecule structures using [AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w) with the freely available [OpenFold3](https://github.com/aqlaboratory/openfold) weights (Apache 2.0 — no commercial restrictions).

MSA generation via the [ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server — **no local databases required**.

Just fill in the **Input sequence(s)** cell and run all. Proteins, nucleic acids and ligands all go in a single box; attention/XLA flags are chosen automatically for your runtime (T4, newer GPUs, or CPU).

**Citations:**
- Abramson et al. (2024) AlphaFold 3. *Nature* [doi:10.1038/s41586-024-07487-w](https://doi.org/10.1038/s41586-024-07487-w)
- Mirdita et al. (2022) ColabFold. *Nature Methods* [doi:10.1038/s41592-022-01488-1](https://doi.org/10.1038/s41592-022-01488-1)
- The OpenFold3 Team (2025) [Github](https://github.com/aqlaboratory/openfold-3) - [doi:10.5281/zenodo.19001000](https://doi.org/10.5281/zenodo.19001000)

**Credits:** AF3 code: Google DeepMind (Apache 2.0) · OF3 weights: AlQuraishi Lab (Apache 2.0)

In [4]:
import platform
import sys

print("Architecture:", platform.machine())
print("Python:", sys.version)
print("Interpreter:", sys.executable)

Architecture: aarch64
Python: 3.12.15 (main, Oct 13 2026, 16:55:48) [GCC 15.3.0]
Interpreter: /home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/bin/python


In [5]:
from pathlib import Path

for name in ("ALPHAFOLD3_READY", "DATA_DONE"):
    Path(name).unlink(missing_ok=True)

In [3]:
#This cell needs only be run once per environment. On future runs, you can select the "AlphaFold 3 (ARM)" kernel from the beginning and skip this cell.
!conda env list

import json
import os
import subprocess

ENV_NAME = "af3-jupyter"

def run(*args, **kwargs):
    subprocess.run(args, check=True, **kwargs)

# Create or update the environment.
envs = json.loads(subprocess.check_output(
    ["conda", "env", "list", "--json"], text=True
))["envs"]

exists = any(os.path.basename(p) == ENV_NAME for p in envs)

run(
    "conda", "install" if exists else "create",
    "-n", ENV_NAME, "-c", "conda-forge", "-y",
    "python=3.12", "pip", "cxx-compiler", "cmake", "ninja", "zlib"
)

prefix = subprocess.check_output(
    ["conda", "run", "-n", ENV_NAME,
     "python", "-c", "import sys; print(sys.prefix)"],
    text=True
).strip()

build_env = os.environ.copy()
build_env["CMAKE_ARGS"] = (
    f"-DZLIB_ROOT={prefix} "
    f"-DZLIB_INCLUDE_DIR={prefix}/include "
    f"-DZLIB_LIBRARY={prefix}/lib/libz.so"
)
build_env["CMAKE_BUILD_PARALLEL_LEVEL"] = "2"

def in_env(*args, **kwargs):
    run("conda", "run", "--no-capture-output",
        "-n", ENV_NAME, *args, **kwargs)

# Build AlphaFold 3 for ARM.
in_env(
    "python", "-m", "pip", "install",
    "ipykernel", "py3Dmol", "py2Dmol", "awscli",
    "git+https://github.com/sokrypton/alphafold3.git@v3.1.5",
    env=build_env
)

in_env("build_data")

# Make this environment available in Jupyter.
in_env(
    "python", "-m", "ipykernel", "install", "--user",
    "--name", ENV_NAME,
    "--display-name", "AlphaFold 3 (ARM)"
)

print("Setup finished. Select Kernel → Change Kernel → AlphaFold 3 (ARM).")


# conda environments:
#
# * -> active
# + -> frozen
af3-jupyter              /home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter
colabfold-jupyter    *   /home/samuelf/.conda-hpc/el9_gh200/envs/colabfold-jupyter
base                     /software/sse2/el9_gh200/easybuild/prefix/software/Miniforge3/26.3.2-2

Channels:
 - conda-forge
Platform: linux-aarch64
Solving environment: done

# All requested packages already installed.

  Cloning https://github.com/sokrypton/alphafold3.git (to revision v3.1.5) to /tmp/pip-req-build-psux3ja0


  Running command git clone --filter=blob:none --quiet https://github.com/sokrypton/alphafold3.git /tmp/pip-req-build-psux3ja0
  Running command git checkout -q 8e77e078129e2a0a7a6f087bef924b4417e1740a


  Resolved https://github.com/sokrypton/alphafold3.git to commit 8e77e078129e2a0a7a6f087bef924b4417e1740a
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Parsing /home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/lib/python3.12/site-packages/share/libcifpp/components.cif
Writing /home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/lib/python3.12/site-packages/alphafold3/constants/converters/ccd.pickle
Done
Loading /home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/lib/python3.12/site-packages/alphafold3/constants/converters/ccd.pickle
Written /home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/lib/python

In [1]:
import sys
print(sys.executable)
import subprocess

import sys
print(sys.executable)
import alphafold3
import jax

print(jax.devices())

subprocess.run([
    "conda", "run", "-n", "af3-jupyter",
    "python", "-m", "ipykernel", "install", "--user",
    "--name", "af3-jupyter",
    "--display-name", "AlphaFold 3 (ARM)"
], check=True)
print("Now please manually select AlphaFold 3 (ARM)")


/home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/bin/python
/home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/bin/python
[CudaDevice(id=0)]
Installed kernelspec af3-jupyter in /home/samuelf/.local/share/jupyter/kernels/af3-jupyter
Now please manually select AlphaFold 3 (ARM)


In [1]:
%%time
#@title Install dependencies (~3 mins)
import os, time, glob

use_af3_weights = True #@param {type:"boolean"}
#@markdown - **use_af3_weights**: leave OFF to use the free, open **OpenFold3** weights. Turn ON to
#@markdown   download the official **AlphaFold 3** weights directly - now public, no login or key required.
#@markdown   Subject to the AF3 weights terms of use.
USE_NATIVE = use_af3_weights
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.17 rdkit==2025.9.4 \
  zstandard awscli tokamax==0.0.11 py3Dmol py2Dmol")
  os.system("pip install -q --no-deps 'https://github.com/sokrypton/alphafold3/releases/download/v3.1.5/alphafold3_open-3.1.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'")
  base = 'https://raw.githubusercontent.com/sokrypton/alphafold3/refs/heads/main'
  for script in ['run_alphafold', 'convert_of3_weights']:
    os.system(f'wget -q -nc {base}/{script}.py')
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights: official AlphaFold 3 (public download) or OpenFold3 (default, free) - both in background
if USE_NATIVE:
  if not os.path.isfile('NATIVE_WEIGHTS_DONE'):
    print('Downloading official AlphaFold 3 weights (public, no login required)...')
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -q -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"; touch NATIVE_WEIGHTS_DONE) &')
elif not os.path.isfile('WEIGHTS_DONE'):
  print('Downloading OpenFold3 weights...')
  os.system('(aws s3 cp s3://openfold/staging/of3-p2-155k.pt . --no-sign-request; touch WEIGHTS_DONE) &')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Building AF3 data files...')
  os.system('(build_data; touch DATA_DONE) &')

# Wait for background jobs
for sentinel in (['NATIVE_WEIGHTS_DONE', 'DATA_DONE'] if USE_NATIVE else ['WEIGHTS_DONE', 'DATA_DONE']):
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

if USE_NATIVE and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

if not USE_NATIVE and not os.path.isfile('af3_converted_weights/of3_ported_weights.bin.zst'):
  print("converting openfold3 weights...")
  os.system("python convert_of3_weights.py --of3_checkpoint of3-p2-155k.pt --output_dir af3_converted_weights")

print('Setup complete!  ' + ('Using official AlphaFold 3 weights.' if USE_NATIVE else 'Using OpenFold3 weights.'))

NATIVE_WEIGHTS_DONE ✓
DATA_DONE ✓
Setup complete!  Using official AlphaFold 3 weights.
CPU times: user 4.93 s, sys: 438 ms, total: 5.37 s
Wall time: 1.4 s


In [2]:
#@title Input sequences
import re, os, json, hashlib
print ("starting cell")

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
#protein = 'RGQRGCEHYDRGCLLKAPCCDKLYTCRLCHDNNEDHQLDRFKVKEVQCINCEKIQHAQQTCEECSTLFGEYYCDICHLFDKDKKQYHCENCGICRIGPKEDFFHCLKCNLCLAMNLQGRHKCIENVSRQNCPICLEDIHTSRVVAHVLPCGHLLHRTCYEEMLKEGYRCPLCMHSAL:MSGSRQAGSGSAGTSPGSSAASSVTSASSSLSSSPSPPSVAVSAAALVSGGVAQAAGSGGLGGPVRPVLVAPAVSGSGGGAVSTGLSRHSCAARPSAGVGGSSSSLGSGSRKRPLLA' #@param {type:"string"}
#human Pirh2 vs. dog 11A Cop1
#protein = 'RGQRGCEHYDRGCLLKAPCCDKLYTCRLCHDNNEDHQLDRFKVKEVQCINCEKIQHAQQTCEECSTLFGEYYCDICHLFDKDKKQYHCENCGICRIGPKEDFFHCLKCNLCLAMNLQGRHKCIENVSRQNCPICLEDIHTSRVVAHVLPCGHLLHRTCYEEMLKEGYRCPLCMHSAL:MSGSRQAGSGSAGTSPGSSAASSVTSASSSLSSSPSPPSAAAAAAAAAAASGGAAQAAGSGGLGGPAARPVLVAAAVSGSGGGAVSAGQSRLSCAARPSAGVGGSSSGLGSGSRKRPLLA'
#canis canis familiaris Pirh2 (uniprot A0A8I3QFQ7) vs. dog 11A Cop1:
protein = 'MAAAAREDGARGREQGLRGCEHYDRGCLLKAPCCDKLYTCRLCHDNNEDHQLDRFKVKEVQCINCEKIQHAQQSCEECSTLFGEYYCSICHLFDKDKKQYHCENCGICRIGPKEDFFHCLKCNLCLAMNLQGKHKCIENVSRQNCPICLEDIHTSRVVAHVLPCGHLLHRTCYEEMLKEGYRCPLCMHSALDMTRYWRQLDDEVAQTPMPSEYQNMTVDILCNDCNRRSTVQFHILGMKCNICESYNTAQAKRISLDQQ:MSGSRQAGSGSAGTSPGSSAASSVTSASSSLSSSPSPPSAAAAAAAAAAASGGAAQAAGSGGLGGPAARPVLVAAAVSGSGGGAVSAGQSRLSCAARPSAGVGGSSSGLGSGSRKRPLLA'
#canis canis familiaris Pirh2 (uniprot A0A8I3QFQ7) vs. dog 9A Cop1:
protein = 'MAAAAREDGARGREQGLRGCEHYDRGCLLKAPCCDKLYTCRLCHDNNEDHQLDRFKVKEVQCINCEKIQHAQQSCEECSTLFGEYYCSICHLFDKDKKQYHCENCGICRIGPKEDFFHCLKCNLCLAMNLQGKHKCIENVSRQNCPICLEDIHTSRVVAHVLPCGHLLHRTCYEEMLKEGYRCPLCMHSALDMTRYWRQLDDEVAQTPMPSEYQNMTVDILCNDCNRRSTVQFHILGMKCNICESYNTAQAKRISLDQQ:MSGSRQAGSGSAGTSPGSSAASSVTSASSSLSSSPSPPSAAAAAAAAASGGAAQAAGSGGLGGPAARPVLVAAAVSGSGGGAVSAGQSRLSCAARPSAGVGGSSSGLGSGSRKRPLLA'


dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')


# Phosphorylate the outer serines bracketing VAVSAAALV.
chain_b = next(
    c["protein"] for c in chains
    if "protein" in c and c["protein"]["id"] == "B"
)

#assert chain_b["sequence"][38:49] == "SVAVSAAALVS", (
#    "Chain B sequence differs: check phosphorylation positions."
#)

#chain_b["modifications"] = [
#    {"ptmType": "SEP", "ptmPosition": 39},
#    {"ptmType": "SEP", "ptmPosition": 49},
#]

# for dog 11A:
#assert chain_b["sequence"][39:51] == "AAAAAAAAAAAS"
#chain_b["modifications"] = [
#    {"ptmType": "SEP", "ptmPosition": 51},
#]

# for dog 9A:
assert chain_b["sequence"][39:49] == "AAAAAAAAAS"
chain_b["modifications"] = [
    {"ptmType": "SEP", "ptmPosition": 49},
]

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input
print ("done with cell")


starting cell
Job "test_d53ba"  ->  results will be written to af3_output/test_d53ba/
done with cell


In [3]:
%%time
#@title Run AlphaFold 3
import sys
import subprocess
import os, shutil, subprocess, glob

from pathlib import Path
from urllib.request import urlopen

%pip install requests

url = (
    "https://raw.githubusercontent.com/sokrypton/alphafold3/"
    "v3.1.5/run_alphafold.py"
)

with urlopen(url, timeout=60) as response:
    script = response.read()

Path("run_alphafold.py").write_bytes(script)

#@markdown Inference settings (defaults match AlphaFold 3 - increase only if needed):
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - `num_recycles`: refinement passes through the network (default 10). More can help large/hard targets, but is slower.
#@markdown - `num_diffusion_samples`: candidate structures generated per seed (default 5). Total models = seeds x samples.

num_recycles = max(1, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-run policy (one folder per job, no timestamped duplicates):
#   overwrite -> wipe this job's folder and recompute
#   skip      -> if a finished result (.cif) is already there, don't recompute
have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
run_it = not (on_existing == 'skip' and have_results)
if run_it:
  shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

# Pick attention impl + XLA flags from the actual device.
# Triton/cuDNN flash attention need Ampere (compute capability >= 8.0);
# 7.x GPUs (T4=7.5, V100=7.0) and CPU use the portable XLA path, and 7.x
# additionally needs the XLA flag that disables the custom-kernel fusion pass.
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []   # extra XLA flags to export for this device (per AlphaFold 3's guidance)

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True
  print('No GPU detected - running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  # T4 / V100 (cc 7.x): XLA attention; disable the custom-kernel fusion pass.
  # (Triton GEMM is not supported on these cards, so it is not disabled here.)
  flash_impl = 'xla'
  xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
  print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
  # L4 / Ada / consumer Ampere (cc 8.6 / 8.9): limited shared memory. XLA's Triton GEMM
  # kernels exceed it ('Shared memory size limit exceeded'), so disable Triton GEMM
  # (falls back to cuBLAS) and use XLA attention to also avoid the Triton attention kernel.
  flash_impl = 'xla'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Ada/consumer GPU (compute capability {cap}) - XLA attention + Triton GEMM disabled (shared-memory limit).')
else:
  # A100 (cc 8.0) and H100 (cc 9.0+): ample shared memory. Triton flash attention,
  # with Triton GEMM disabled per AlphaFold 3's recommended XLA_FLAGS.
  flash_impl = 'triton'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Datacenter GPU (compute capability {cap}) - Triton flash attention + Triton GEMM disabled.')

# Export XLA flags so the child shell (and JAX inside it) inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

# Weights: use the official AlphaFold 3 weights if they were fetched, else OpenFold3.
if os.path.isdir('af3_native_weights') and glob.glob('af3_native_weights/*.bin*'):
  model_dir, use_of3 = 'af3_native_weights', False
  print('Weights: official AlphaFold 3')
else:
  model_dir, use_of3 = 'af3_converted_weights', True
  print('Weights: OpenFold3')

cmd = [
    sys.executable, 'run_alphafold.py',
    f'--json_path={json_path}',
    f'--model_dir={model_dir}',
    '--norun_data_pipeline',
    f'--output_dir={OUTPUT_DIR}',
    '--cache_dir=/tmp/af3_cache',
    '--force_output_dir',          # reuse af3_output/<jobname>/ instead of a timestamped copy
    f'--flash_attention_implementation={flash_impl}',
    f'--num_recycles={num_recycles}',
    f'--num_diffusion_samples={num_diffusion_samples}',
]
if msa_mode == 'mmseqs2_server':
  cmd.append('--use_msa_server')
if nojit:
  cmd.append('--nojit')
if use_of3:
  cmd.append('--of3_weights')

if run_it:
  print(" ".join(cmd))
  subprocess.run(cmd, check=True)
  print(f'\nDone -> {job_dir}/')
else:
  print(f'Skipping: results already exist in {job_dir}/  (set on_existing=overwrite to recompute).')
print("Done with cell")

Note: you may need to restart the kernel to use updated packages.
Datacenter GPU (compute capability 9.0) - Triton flash attention + Triton GEMM disabled.
XLA_FLAGS = --xla_gpu_enable_triton_gemm=false
Weights: official AlphaFold 3
/home/samuelf/.conda-hpc/el9_gh200/envs/af3-jupyter/bin/python run_alphafold.py --json_path=/tmp/af3_inputs/test_d53ba.json --model_dir=af3_native_weights --norun_data_pipeline --output_dir=af3_output --cache_dir=/tmp/af3_cache --force_output_dir --flash_attention_implementation=triton --num_recycles=10 --num_diffusion_samples=5 --use_msa_server


W1023 13:00:54.053074 70369210812928 features.py:1423] test_d53ba, random_seed=1 token 307 (AtomLayout(atom_name=array([['N', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '',
        '', '', '', '', '', '', '', '']], dtype=object), res_id=array([[49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49,
        49, 49, 49, 49, 49, 49, 49, 49]]), chain_id=array([['B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
        'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B']],
      dtype=object), atom_element=array([['N', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '',
        '', '', '', '', '', '', '', '']], dtype=object), res_name=array([['SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP',
        'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP',
        'SEP', 'SEP', 'SEP', 'SEP', 'SEP', 'SEP']], dtype=object), chain_type=array([['polypeptide(L)', 'polypeptide(L)', 'polypeptide(L)',
        'polypeptide(L)', 'polype


Running AlphaFold 3. Please note that standard AlphaFold 3 model parameters are
only available under terms of use provided at
https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md.
If you do not agree to these terms and are using AlphaFold 3 derived model
parameters, cancel execution of AlphaFold 3 inference with CTRL-C, and do not
use the model parameters.

Found local GPU devices: [CudaDevice(id=0)], using device 0: cuda:0
Building model from scratch...
Checking that model parameters can be loaded...
Querying ColabFold MSA server for 2 unique protein sequence(s)...
MSA query complete.
Querying ColabFold for paired MSA (2 unique protein chains)...

Running fold job test_d53ba...
Output will be written in af3_output/test_d53ba
Skipping data pipeline...
Writing model input JSON to af3_output/test_d53ba/test_d53ba_data.json
Predicting 3D structure for test_d53ba with 1 seed(s)...
Featurising data with 1 seed(s)...
Featurising data with seed 1.
Featurising data 

In [4]:
#Inventory cell
from pathlib import Path
import json, re, hashlib

print("Job | PIRH2 length/hash | polyA | modifications B | models")

for folder in sorted(Path("af3_output").iterdir()):
    inputs = list(folder.glob("*_data.json"))
    if not inputs:
        continue

    data = json.loads(inputs[0].read_text())
    proteins = {
        p["id"]: p
        for entity in data["sequences"]
        if (p := entity.get("protein")) and isinstance(p["id"], str)
    }
    if not {"A", "B"} <= proteins.keys():
        continue

    a, b = proteins["A"], proteins["B"]
    a_hash = hashlib.sha1(a["sequence"].encode()).hexdigest()[:8]
    tract = re.search(r"S(A{5,})S", b["sequence"])
    n_ala = len(tract.group(1)) if tract else "?"
    mods = [(m["ptmPosition"], m["ptmType"])
            for m in b.get("modifications", [])]
    count = len(list(folder.glob("seed-*_sample-*/*_model.cif")))

    print(f"{folder.name} | {len(a['sequence'])}/{a_hash}"
          f" | {n_ala}A | {mods} | {count}")

Job | PIRH2 length/hash | polyA | modifications B | models
test_07122 | 177/a7c2cefc | 11A | [(51, 'SEP')] | 5
test_2d467 | 177/a7c2cefc | ?A | [(39, 'SEP'), (49, 'SEP')] | 4
test_93af8 | 259/f999beb3 | 11A | [(51, 'SEP')] | 5
test_d53ba | 259/f999beb3 | 9A | [(49, 'SEP')] | 5


In [5]:
#comparison cell
import json, re
from pathlib import Path
from collections import defaultdict
import numpy as np
import gemmi

jobs = {"11A": "test_93af8", "9A": "test_d53ba"}

def read_input(job):
    file = next(Path("af3_output", job).glob("*_data.json"))
    data = json.loads(file.read_text())
    proteins = [e["protein"] for e in data["sequences"] if "protein" in e]
    return data, proteins

d11, p11 = read_input(jobs["11A"])
d9, p9 = read_input(jobs["9A"])
assert p11[0]["sequence"] == p9[0]["sequence"], "PIRH2 differs"
normalize = lambda s: re.sub(r"A{5,}", "<polyA>", s)
assert normalize(p11[1]["sequence"]) == normalize(p9[1]["sequence"]), \
    "COP1 differs outside the polyA tract"
assert d11["modelSeeds"] == d9["modelSeeds"], "Seeds differ"

comparison = {}

for label, job in jobs.items():
    phosphate_position = 51 if label == "11A" else 49
    contacts = defaultdict(list)
    metrics = defaultdict(list)
    files = sorted(Path("af3_output", job).glob(
        "seed-*_sample-*/*_model.cif"
    ))

    for file in files:
        model = gemmi.read_structure(str(file))[0]
        residues_a = [r for c in model if c.name == "A" for r in c]
        residues_b = {r.seqid.num: r for c in model
                      if c.name == "B" for r in c}
        assert residues_b[39].name == "SER"
        assert residues_b[phosphate_position].name == "SEP"

        conf_file = next(p for p in file.parent.glob("*_confidences.json")
                         if "summary" not in p.name)
        conf = json.loads(conf_file.read_text())
        pae = np.asarray(conf["pae"])
        ids = list(zip(conf["token_chain_ids"],
                       map(int, conf["token_res_ids"])))
        ai = [i for i, (c, r) in enumerate(ids) if c == "A"]

        summary = json.loads(next(
            file.parent.glob("*_summary_confidences.json")
        ).read_text())
        if summary.get("iptm") is not None:
            metrics["ipTM"].append(summary["iptm"])

        groups = {
            "phosphate": (phosphate_position,
                          {"O1P", "O2P", "O3P", "OP1", "OP2", "OP3"}),
            "S39 backbone": (39, {"N", "O"}),
        }

        for group, (position, names) in groups.items():
            target = [a for a in residues_b[position] if a.name in names]
            assert target, f"Missing atoms for {group}"
            bi = [i for i, pair in enumerate(ids)
                  if pair == ("B", position)]

            # Keep the two PAE directions separate.
            metrics[group + " PAE A/B"].append(
                float(pae[np.ix_(ai, bi)].mean()))
            metrics[group + " PAE B/A"].append(
                float(pae[np.ix_(bi, ai)].mean()))

            for residue in residues_a:
                # Phosphate: nearby nitrogen donors.
                # S39 backbone: nearby polar atoms (candidate contacts).
                atoms = [a for a in residue if a.element.name in
                         (("N",) if group == "phosphate" else ("N", "O"))]
                if atoms:
                    distance = min(t.pos.dist(a.pos)
                                   for t in target for a in atoms)
                    key = (group, f"{residue.name}{residue.seqid.num}")
                    contacts[key].append(distance)

    comparison[label] = {"metrics": metrics, "contacts": contacts}
    print(f"\n{label}: {len(files)} models")
    for name, values in metrics.items():
        print(f"  {name}: median {np.median(values):.2f}"
              f"  range {min(values):.2f}–{max(values):.2f}")

    for group in ("phosphate", "S39 backbone"):
        candidates = [(res, ds) for (g, res), ds in contacts.items()
                      if g == group]
        candidates.sort(key=lambda x: (
            -sum(d <= 3.5 for d in x[1]), np.median(x[1])))
        print(f"  {group}: most recurrent close approaches")
        for residue, distances in candidates[:3]:
            count = sum(d <= 3.5 for d in distances)
            print(f"    {residue}: {count}/{len(distances)} within 3.5 Å;"
                  f" distances {np.round(distances, 2).tolist()}")


11A: 5 models
  ipTM: median 0.45  range 0.35–0.48
  phosphate PAE A/B: median 8.85  range 8.48–9.05
  phosphate PAE B/A: median 17.45  range 16.03–19.52
  S39 backbone PAE A/B: median 14.37  range 13.81–15.18
  S39 backbone PAE B/A: median 25.32  range 24.69–26.21
  phosphate: most recurrent close approaches
    ARG54: 4/5 within 3.5 Å; distances [1.77, 2.06, 2.31, 1.89, 3.51]
    ARG41: 2/5 within 3.5 Å; distances [3.46, 3.51, 2.36, 4.44, 4.57]
    CYS108: 0/5 within 3.5 Å; distances [5.22, 4.57, 4.72, 5.15, 4.15]
  S39 backbone: most recurrent close approaches
    ARG195: 0/5 within 3.5 Å; distances [7.93, 7.05, 7.68, 9.58, 3.83]
    ASP192: 0/5 within 3.5 Å; distances [8.48, 8.42, 8.29, 9.02, 4.58]
    TYR173: 0/5 within 3.5 Å; distances [8.83, 9.01, 8.45, 6.81, 7.2]

9A: 5 models
  ipTM: median 0.54  range 0.52–0.55
  phosphate PAE A/B: median 8.57  range 8.31–8.69
  phosphate PAE B/A: median 14.85  range 14.50–15.02
  S39 backbone PAE A/B: median 13.25  range 12.93–13.79
  S39 b

In [ ]:
from pathlib import Path
import csv, json
import numpy as np
import py2Dmol

folder = Path(job_dir)
ranking = list(folder.glob("*ranking_scores.csv"))

if ranking:
    with ranking[0].open() as f:
        rows = sorted(csv.DictReader(f),
                      key=lambda r: float(r["ranking_score"]), reverse=True)
    cifs = [
        next((folder / f"seed-{r['seed']}_sample-{r['sample']}").glob("*_model.cif"))
        for r in rows
    ]
else:
    cifs = sorted(folder.glob("seed-*_sample-*/*_model.cif"))

assert cifs, f"No models found in {folder}"

viewer = py2Dmol.view(size=(500, 500), pae=True, autoplay=True)

for cif in cifs:
    conf_file = next(
        p for p in cif.parent.glob("*_confidences.json")
        if "summary" not in p.name
    )
    pae = np.asarray(json.loads(conf_file.read_text())["pae"])
    viewer.add_pdb(str(cif), name="models", paes=pae)

print(f"Loaded {len(cifs)} models — use play or the slider.")
viewer.show()

In [7]:
#Get contacts between chains
import csv, json
from pathlib import Path
import gemmi
import numpy as np

rows = []
for cif in cifs:
    path = Path(cif)
    conf_path = next(
        p for p in path.parent.glob("*_confidences.json")
        if "summary" not in p.name
    )
    conf = json.loads(conf_path.read_text())
    pae = np.asarray(conf["pae"])
    ids = list(zip(conf["token_chain_ids"],
                   map(int, conf["token_res_ids"])))

    def indices(chain, residue):
        return [i for i, pair in enumerate(ids)
                if pair == (chain, residue)]

    model = gemmi.read_structure(str(path))[0]
    arginines = [r for c in model if c.name == "A"
                 for r in c if r.name == "ARG"]
    phosphoserines = [
        r for c in model if c.name == "B"
        for r in c
        if r.name == "SEP" and 38 <= r.seqid.num <= 51
    ]
    for ser in phosphoserines:
        oxygens = [a for a in ser
                   if a.name in {"O1P", "O2P", "O3P",
                                 "OP1", "OP2", "OP3"}]
        if not oxygens:
            raise ValueError("Phosphate oxygen names not recognised.")

        for arg in arginines:
            nitrogens = [a for a in arg
                         if a.name in {"NE", "NH1", "NH2"}]
            distance, nitrogen, oxygen = min(
                (n.pos.dist(o.pos), n.name, o.name)
                for n in nitrogens for o in oxygens
            )
            ai = indices("A", arg.seqid.num)
            bi = indices("B", ser.seqid.num)
            if not ai or not bi:
                raise ValueError("Cannot map residue numbers to PAE tokens.")

            rows.append({
                "model": path.parent.name,
                "Arg_A": arg.seqid.num,
                "pSer_B": ser.seqid.num,
                "atoms": f"{nitrogen}-{oxygen}",
                "distance_A": round(distance, 2),
                "PAE_A_B": round(float(pae[np.ix_(ai, bi)].mean()), 2),
                "PAE_B_A": round(float(pae[np.ix_(bi, ai)].mean()), 2),
            })

if not rows:
    raise ValueError("No B39/B49 phosphoserines found; check the loaded job.")

output = Path(job_dir) / "arginine_phosphoserine_contacts.csv"
with output.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print("Closest three arginines per phosphoserine in each model:")
for model, ser in sorted({(r["model"], r["pSer_B"]) for r in rows}):
    group = [r for r in rows
             if r["model"] == model and r["pSer_B"] == ser]
    for r in sorted(group, key=lambda r: r["distance_A"])[:3]:
        print(r)
print("Full table:", output)

Closest three arginines per phosphoserine in each model:
{'model': 'seed-1_sample-0', 'Arg_A': 54, 'pSer_B': 49, 'atoms': 'NH1-O3P', 'distance_A': 2.72, 'PAE_A_B': 3.95, 'PAE_B_A': 6.43}
{'model': 'seed-1_sample-0', 'Arg_A': 41, 'pSer_B': 49, 'atoms': 'NE-O1P', 'distance_A': 3.93, 'PAE_A_B': 3.48, 'PAE_B_A': 6.72}
{'model': 'seed-1_sample-0', 'Arg_A': 25, 'pSer_B': 49, 'atoms': 'NH2-O2P', 'distance_A': 3.99, 'PAE_A_B': 4.57, 'PAE_B_A': 8.42}
{'model': 'seed-1_sample-1', 'Arg_A': 41, 'pSer_B': 49, 'atoms': 'NE-O2P', 'distance_A': 2.97, 'PAE_A_B': 3.53, 'PAE_B_A': 6.63}
{'model': 'seed-1_sample-1', 'Arg_A': 54, 'pSer_B': 49, 'atoms': 'NE-O2P', 'distance_A': 2.97, 'PAE_A_B': 4.16, 'PAE_B_A': 6.6}
{'model': 'seed-1_sample-1', 'Arg_A': 25, 'pSer_B': 49, 'atoms': 'NH2-O3P', 'distance_A': 4.44, 'PAE_A_B': 4.72, 'PAE_B_A': 8.49}
{'model': 'seed-1_sample-2', 'Arg_A': 41, 'pSer_B': 49, 'atoms': 'NH1-O2P', 'distance_A': 1.54, 'PAE_A_B': 3.43, 'PAE_B_A': 6.78}
{'model': 'seed-1_sample-2', 'Arg_A':

In [8]:
#@title Quality metrics and plots
import json, os
import numpy as np
import matplotlib.pyplot as plt

conf_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_confidences.json'
summ_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_summary_confidences.json'

with open(conf_path) as f:
  conf = json.load(f)
with open(summ_path) as f:
  summ = json.load(f)

plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))   # pLDDT is per-ATOM
token_chain_ids = conf.get('token_chain_ids', [])                                # PAE is per-TOKEN
pae = np.array(conf.get('pae', []), dtype=float)

# ── Summary (ipTM is None for single-chain jobs — guard before formatting) ─
def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

mean_plddt = summ.get('mean_plddt')
if mean_plddt is None and plddts.size:
  mean_plddt = float(np.mean(plddts))
iptm = summ.get('iptm')

print('=' * 38)
print(f'Mean pLDDT     : {fmt(mean_plddt)}')
print(f'pTM            : {fmt(summ.get("ptm"))}')
print(f'ipTM           : {fmt(iptm)}' + ('   (single chain — no interface)' if iptm is None else ''))
print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
print('=' * 38)

# ── Plots ───────────────────────────────────────────────────
has_pae = pae.ndim == 2 and pae.size > 0
ncols = 2 if has_pae else 1
fig, axes = plt.subplots(1, ncols, figsize=(13 if has_pae else 6.5, 4))
axes = np.atleast_1d(axes)

# pLDDT per residue — a line (coloured per chain when there is more than one)
ax = axes[0]
x = np.arange(len(plddts))
xmax = max(len(plddts) - 1, 1)
ax.set_xlim(0, xmax)
ax.set_ylim(0, 100)

unique_chains = list(dict.fromkeys(plddt_chain_ids))
if len(unique_chains) > 1:
  colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
  tcid = np.array(plddt_chain_ids)
  for ch, col in zip(unique_chains, colors):
    y = np.where(tcid == ch, plddts, np.nan)   # NaN gaps keep chains as separate lines
    ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
  for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
    ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
  ax.legend(loc='lower right', fontsize=8)
else:
  ax.plot(x, plddts, lw=1.5, color='#1f77b4')

for y in (50, 70, 90):
  ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
  ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
ax.set_xlabel('Atom')
ax.set_ylabel('pLDDT')
ax.set_title('Predicted pLDDT per atom')

# PAE matrix
if has_pae:
  ax = axes[1]
  im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
  plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
  if token_chain_ids:
    for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
      ax.axhline(b - 0.5, c='black', lw=0.8)
      ax.axvline(b - 0.5, c='black', lw=0.8)
  ax.set_xlabel('Scored residue')
  ax.set_ylabel('Aligned residue')
  ax.set_title('Predicted Aligned Error (PAE)')

plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'

In [9]:
import json
from pathlib import Path
import gemmi
import numpy as np
from collections import defaultdict

results = defaultdict(list)

for cif in cifs:
    path = Path(cif)
    model = gemmi.read_structure(str(path))[0]
    ser = next(r for c in model if c.name == "B"
               for r in c if r.seqid.num == 39)
    assert ser.name == "SER", f"B39 is {ser.name}, not unmodified SER"
    og = next(a for a in ser if a.name == "OG")

    conf_path = next(p for p in path.parent.glob("*_confidences.json")
                     if "summary" not in p.name)
    conf = json.loads(conf_path.read_text())
    pae = np.asarray(conf["pae"])
    ids = list(zip(conf["token_chain_ids"],
                   map(int, conf["token_res_ids"])))

    def idx(chain, number):
        return [i for i, pair in enumerate(ids)
                if pair == (chain, number)]

    bi = idx("B", 39)
    nearby = []

    for chain in model:
        if chain.name != "A":
            continue
        for residue in chain:
            atoms = [a for a in residue
                     if a.element.name in ("N", "O")]
            ai = idx("A", residue.seqid.num)
            if not atoms or not ai or not bi:
                continue

            distance, atom = min(
                (og.pos.dist(a.pos), a.name) for a in atoms
            )
            ab = float(pae[np.ix_(ai, bi)].mean())
            ba = float(pae[np.ix_(bi, ai)].mean())
            label = f"{residue.name}{residue.seqid.num}"
            results[label].append((distance, ab, ba))
            nearby.append((distance, label, atom, ab, ba))

    print(f"\n{path.parent.name}: closest to B39 OG")
    for d, label, atom, ab, ba in sorted(nearby)[:3]:
        print(f"  {label}:{atom}  {d:.2f} Å"
              f"  PAE A/B={ab:.1f}, B/A={ba:.1f} Å")

print("\nRecurring close approaches (OG–N/O ≤3.5 Å):")
for label, values in sorted(
    results.items(),
    key=lambda item: (-sum(v[0] <= 3.5 for v in item[1]),
                      np.mean([v[0] for v in item[1]]))
):
    count = sum(v[0] <= 3.5 for v in values)
    if count:
        print(f"  {label}: {count}/{len(values)} models;"
              f" distances {[round(v[0], 2) for v in values]} Å")


seed-1_sample-0: closest to B39 OG
  TYR173:OH  3.77 Å  PAE A/B=9.3, B/A=14.6 Å
  HIS160:NE2  4.16 Å  PAE A/B=9.1, B/A=12.6 Å
  ALA190:O  5.70 Å  PAE A/B=9.1, B/A=15.3 Å

seed-1_sample-2: closest to B39 OG
  HIS160:NE2  6.03 Å  PAE A/B=9.6, B/A=12.2 Å
  ASN129:ND2  6.84 Å  PAE A/B=9.5, B/A=13.5 Å
  TYR173:OH  7.21 Å  PAE A/B=10.2, B/A=14.9 Å

seed-1_sample-3: closest to B39 OG
  HIS160:NE2  5.70 Å  PAE A/B=9.2, B/A=11.6 Å
  ASN129:ND2  6.84 Å  PAE A/B=9.0, B/A=12.5 Å
  TYR173:OH  7.04 Å  PAE A/B=9.6, B/A=13.8 Å

seed-1_sample-1: closest to B39 OG
  ASN129:ND2  4.47 Å  PAE A/B=10.1, B/A=13.1 Å
  ARG170:NH2  7.16 Å  PAE A/B=9.7, B/A=15.5 Å
  HIS160:ND1  8.02 Å  PAE A/B=10.2, B/A=13.4 Å

seed-1_sample-4: closest to B39 OG
  ASP192:N  6.17 Å  PAE A/B=10.3, B/A=12.9 Å
  TYR173:OH  6.77 Å  PAE A/B=10.3, B/A=15.1 Å
  HIS160:NE2  7.14 Å  PAE A/B=9.8, B/A=13.4 Å

Recurring close approaches (OG–N/O ≤3.5 Å):


In [10]:
import py3Dmol
from pathlib import Path

v = py3Dmol.view(width=700, height=500)
v.addModel(Path(cifs[2]).read_text(), "mmcif")
v.setStyle({}, {"cartoon": {"color": "lightgrey"}})
v.addStyle({"chain": "B"},
           {"cartoon": {"color": "cyan"}})
#            "stick": {"color": "red"}})

v.addStyle(
    {"chain": "B", "resi": list(range(38, 52)), "resn": ["SER", "SEP"]},
    {"sphere": {"color": "red", "scale": 0.6},
     "stick": {"color": "red"}}
)

v.addStyle({"chain": "A"},
           {"stick": {"colorscheme": "Jmol", "radius": 0.15}})
labels = {"fontSize": 12, "backgroundOpacity": 0.6}

v.addResLabels(
    {"chain": "B", "resi": list(range(38, 52)), "resn": ["SER", "SEP"]}, labels
    #{"chain": "B", "resi": [51, 51], "atom": "CA"}, labels
)
v.addResLabels(
    {"chain": "A", "resn": "ARG", "atom": "CA"}, labels
)
v.zoomTo()
v.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
#@title Download results
from google.colab import files
import os

results_zip = f'{jobname}.result.zip'
os.system(f'zip -r {results_zip} {OUTPUT_DIR}/{jobname}')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start**
1. Fill in the sequence(s) in the **Input sequence(s)** cell.
2. Press **Runtime → Run all**.
3. The install cell (first run only) downloads OpenFold3 weights and builds AF3 data files in the background — subsequent runs reuse them.

---

## Model weights

By default the notebook uses the open **OpenFold3** weights (Apache-2.0, no restrictions). To run the **official AlphaFold 3** weights instead, tick the **use_af3_weights** box in the install cell. The weights are now publicly downloadable (no login, key, or DeepMind approval required) and are fetched directly into `af3_native_weights/`, then used automatically (the run cell detects them and drops the `--of3_weights` flag). Leaving **use_af3_weights** off keeps the OpenFold3 path.

Official AlphaFold 3 parameters are subject to the [AF3 weights terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md); run_alphafold prints a reminder of these terms when they are used.

---

## Sequence input

Each molecule type has its own box. Within a box, separate multiple chains with `:`.

| Box | What goes in it | Example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Chains are assigned IDs A, B, C, … following AlphaFold 3's canonical order (protein → RNA → DNA → ligand; CCD ligands before SMILES ligands). Mix freely across boxes to build a complex — e.g. a protein in **protein**, `AUGCAUGC` in **rna**, and `ATP` in **ligand_ccd**.

- **Homo-oligomers**: identical protein sequences are merged automatically, so `SEQ:SEQ` = homodimer, `SEQ:SEQ:SEQ` = homotrimer.
- Protein / DNA / RNA sequences and CCD codes are upper-cased automatically; **SMILES are left exactly as typed** (case is meaningful in SMILES).
- Spaces and newlines inside an entry are ignored, and **extra colons are forgiven** — `SEQ1::::SEQ2` is the same as `SEQ1:SEQ2`. Leave a box empty if unused.
- *Note:* because `:` separates entries, an atom-mapped SMILES that itself contains a colon (e.g. `[C:1]`) isn't supported via the box — use a raw AF3 JSON for that edge case.

## Seeds

Enter one or more model seeds in the **seeds** box, comma-separated (e.g. `1,2,3`). Each seed is an independent prediction (more seeds = more sampling, more runtime). Non-numeric characters are ignored and duplicates are dropped, so `1, 1, foo, 7` becomes seeds `1` and `7`.

## MSA modes

- **`mmseqs2_server`** *(recommended)*: queries the public [ColabFold](https://colabfold.mmseqs.com/) MMseqs2 API. Covers UniRef30 + environmental sequences for proteins. RNA/DNA chains always run MSA-free (ColabFold is protein-only).
- **`single_sequence`**: no MSA, query sequence only. Faster but less accurate, especially for monomers with close homologs.

## Output files (inside the downloaded zip)

| File | Contents |
|---|---|
| `*.cif` | Best-ranked structure in mmCIF format. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probs. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Ranking scores for all seed × sample combinations. |
| `seed-N_sample-M/` | Individual prediction directories (one per seed/sample). |
| `TERMS_OF_USE.md` | Apache 2.0 notice (OpenFold3 weights have no commercial restrictions). |

## Interpreting confidence scores

- **pLDDT > 90**: very high confidence.
- **pLDDT 70–90**: confident, backbone generally reliable.
- **pLDDT 50–70**: low confidence, treat with caution.
- **pLDDT < 50**: very low, likely disordered or incorrect.
- **PAE**: lower values = confident relative positioning between residue pairs. Useful for assessing interface quality in complexes.
- **ipTM > 0.8**: strong evidence for a well-defined complex interface. **ipTM is `n/a` for single-chain jobs** (there is no interface to score).

## Troubleshooting

- **Check runtime type**: `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 are faster).
- **OOM error**: reduce sequence length or use a larger-memory GPU runtime.
- **MSA server timeout**: the public ColabFold server is rate-limited. Try again later or switch to `single_sequence` mode.
- **Download popup blocked**: disable your ad blocker.
- **Weight download slow**: the OF3 checkpoint is ~1.4 GB from AWS S3; the install cell waits automatically.

## License

AlphaFold 3 source code and OpenFold3 weights are both licensed under [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0). Outputs generated with OpenFold3 weights are **not** subject to Google DeepMind's AlphaFold 3 Output Terms of Use and may be used freely, including commercially.

## Bugs / feedback

Report issues at https://github.com/sokrypton/alphafold3/issues